# SedTRAILS connectivity analysis

This notebook compiles source-to-sink adjacency matrices from SedTRAILS trajectory NetCDF files and analyzes them with NetworkX. See the NetworkX documentation for graph algorithms and metrics: https://networkx.org/documentation/stable/.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt

from sedtrails.pathway_visualizer.trajectories import read_netcdf
from sedtrails.pathway_visualizer.sedtrails_plotting import load_from_xarray
from sedtrails.simulation_analysis.connectivity import (
    adjacency_to_digraph,
    compile_adjacency_matrix,
    compute_network_metrics,
    compute_node_metrics,
    generate_connectivity_polygons,
    plot_adjacency_matrix,
    plot_network_geographic,
    plot_network_layout,
    read_adjacency_netcdf,
    write_adjacency_netcdf,
)
from sedtrails.simulation_analysis.connectivity.polygons import source_positions_from_arrays

results_file = Path('../results/sedtrails_results.nc')
ds = read_netcdf(results_file)
tr = load_from_xarray(ds)
source_xy = source_positions_from_arrays(tr.x, tr.y, tr.valid_mask())
source_xy[:5]

In [ ]:
# One cell per initial particle/source position.
connectivity = generate_connectivity_polygons(source_xy, mode='per_source')

# Alternative: aggregate initial positions into n cells.
# connectivity = generate_connectivity_polygons(source_xy, mode='n_cells', n_cells=8)

In [ ]:
# Defaults: mode='all', raw particle-position counts, repeated visits counted, self-links included.
adjacency_ds = compile_adjacency_matrix(tr, connectivity)
adjacency_file = results_file.with_name('sedtrails_connectivity_adjacency.nc')
write_adjacency_netcdf(adjacency_ds, adjacency_file)
adjacency_ds

In [ ]:
# Read an existing adjacency file instead of recompiling.
adjacency_ds = read_adjacency_netcdf(adjacency_file)
graph = adjacency_to_digraph(adjacency_ds)
compute_network_metrics(graph)

In [ ]:
node_metrics = compute_node_metrics(graph)
node_metrics.keys()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot_adjacency_matrix(adjacency_ds, log_scale=False, ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot_network_geographic(graph, ax=ax)
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
plot_network_layout(graph, layout='spring', ax=ax)
plt.show()

In [ ]:
# Other compile modes:
adj_final = compile_adjacency_matrix(tr, connectivity, mode='final')
adj_time = compile_adjacency_matrix(tr, connectivity, mode='time')

# Other count/weight options:
adj_unique = compile_adjacency_matrix(tr, connectivity, count_repeated_visits=False)
adj_probability = compile_adjacency_matrix(tr, connectivity, weight='probability')
adj_no_self = compile_adjacency_matrix(tr, connectivity, include_self_links=False)